# TeachAgent 用户视角演示

这份 notebook 不看底层大 JSON，只看用户真正会感受到的结果。

它会按 4 个部分演示：

1. 系统里现在有哪些知识点卡片可用
2. 一道题进来后，系统会推荐哪些知识点，学生最后如何确认
3. 学生开始复习时，第一页会看到什么，以及按钮反馈后会怎样变化
4. 系统如何形成长期画像，并轻量影响讲解方式与复习排序

默认只使用 demo 数据，不会修改正式知识库。

In [ ]:
import copy
import json
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path

PROJECT_ROOT = Path('/Users/xumuchi/Desktop/TeachAgent')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from annotation_to_review_state import convert_annotations_to_review_state
from coach_agent import (
    CoachSession,
    ErrorType,
    analyze_student_reply,
    apply_student_memory_bias,
    build_reply_analysis_fallback,
    build_student_memory_context,
    build_student_memory_strategy_note,
    get_coach_strategy,
    infer_completed_from_reply,
)
from review_bundle_builder import build_review_bundles, load_example_map, load_leaf_card_lookup
from review_scheduler import build_review_schedule
from review_state_manager import apply_review_action
from scripts.convert_leaf_draft_to_cards import load_inventory
from student_memory_events import (
    binding_result_to_memory_event,
    coach_response_to_memory_event,
    diagnosis_result_to_memory_event,
    review_state_update_to_memory_event,
    student_choice_to_memory_event,
)
from student_memory_store import append_event, build_profile_from_store
from wrong_question_binder import WrongQuestionBinder, summarize_result

BASE_TIME = datetime(2026, 6, 24, 20, 0, tzinfo=timezone(timedelta(hours=8)))
DEMO_NOW = datetime(2026, 6, 24, 21, 0, tzinfo=timezone(timedelta(hours=8)))
STUDENT_ID = 'student_user_walkthrough_demo'
SESSION_ID = 'session_user_walkthrough_demo_001'

SCRATCH_DIR = PROJECT_ROOT / 'scratch' / 'teachagent_user_walkthrough'
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
MEMORY_EVENTS_JSONL = SCRATCH_DIR / 'student_memory_events_demo.jsonl'

EXAMPLES_MD_PATH = PROJECT_ROOT / 'docs' / 'rag_samples' / 'taizhou_simulated_exam_examples_batch_01.md'

def format_path(path):
    return ' > '.join(path or [])

def reset_demo_file(path: Path):
    if path.exists():
        path.unlink()

def find_primary_card(node_id: str, leaf_card_lookup: dict[str, list[dict]]) -> dict | None:
    rows = leaf_card_lookup.get(node_id, [])
    if not rows:
        return None
    for row in rows:
        if row.get('is_primary'):
            return row
    return rows[0]

def print_leaf_card_preview(node_id: str, inventory: dict[str, dict], leaf_card_lookup: dict[str, list[dict]]) -> None:
    meta = inventory[node_id]
    card = find_primary_card(node_id, leaf_card_lookup)
    print(f'知识点：{meta["name"]}')
    print(f'路径：{format_path(meta.get("path"))}')
    print(f'卡片类型：{card.get("card_type") if card else "无"}')
    print(f'复习提示：{card.get("review_cue") if card else "无"}')
    if card and card.get('text'):
        print(f'内容预览：{card["text"][:110]}...')
    print()

def short_stem(text: str, limit: int = 42) -> str:
    text = ' '.join((text or '').split())
    return text if len(text) <= limit else text[:limit] + '...'

def print_binder_case(case_name: str, question_payload: dict, result_summary: dict, top_candidates: list[dict], selection: dict) -> None:
    print(f'题目：{case_name}')
    print(f'题干：{short_stem(question_payload.get("stem", ""), 60)}')
    print('系统推荐 Top-3：')
    for item in top_candidates[:3]:
        print(f'  - {item["title"]} | 分数 {item["bind_score"]}')
    print(f'系统主推荐：{result_summary["primary_node_id"]}')
    print(f'学生最终主知识点：{selection["primary_node_ids"][0]}')
    print(f'学生补充辅助知识点：{selection["secondary_node_ids"]}')
    print()

def summarize_bundle(bundle: dict) -> str:
    if bundle.get('mode') == 'leaf_first':
        title = (((bundle.get('leaf_card') or {}).get('title')) or bundle.get('node_id'))
        count = bundle.get('question_count') or 0
        return f'{title} | 配套题 {count} 道 | {bundle.get("bundle_reason")}'
    question = bundle.get('question') or {}
    stem = short_stem((question.get('content') or {}).get('stem', ''), 42)
    linked_titles = [item.get('title') for item in bundle.get('linked_leaf_cards', []) if item.get('title')]
    return f'{stem} | 关联知识点：{linked_titles} | {bundle.get("bundle_reason")}'

def summarize_node_ranking(schedule, limit: int = 3) -> list[str]:
    return [
        f'{index}. {item["node_id"].split(".")[-1]} | 分数 {item["priority_score"]} | 记忆加成 {item.get("memory_priority_boost", 0)}'
        for index, item in enumerate(schedule.node_priorities[:limit], start=1)
    ]

def summarize_question_ranking(schedule, limit: int = 3) -> list[str]:
    return [
        f'{index}. {item["question_id"]} | 分数 {item["priority_score"]} | 最近结果 {item.get("last_result")} | 记忆加成 {item.get("memory_priority_boost", 0)}'
        for index, item in enumerate(schedule.question_priorities[:limit], start=1)
    ]

def teaching_mode_label(value: str) -> str:
    mapping = {
        'concept_first': '先讲概念',
        'strategy_first': '先点中间目标和路线',
        'condition_first': '先回到题干条件',
        'checklist_first': '先补检查流程',
        'balanced': '平衡模式',
    }
    return mapping.get(value, value)

def review_mode_label(value: str) -> str:
    mapping = {
        'leaf_first': '先知识点后题目',
        'question_first': '先题目后知识点',
        'mixed': '混合模式',
    }
    return mapping.get(value, value)


## 1. 当前系统里有什么知识点卡片

这里直接看三张代表性卡片：数列、向量、解析几何。

In [ ]:
INVENTORY = load_inventory()
LEAF_CARD_LOOKUP = load_leaf_card_lookup()

SAMPLE_NODE_IDS = [
    'math.数列与不等式.数列.递推数列转化方法.构造等比数列',
    'math.三角函数_平面向量与解三角形.平面向量.向量夹角公式',
    'math.解析几何.直线的方程.点到直线的距离公式',
]

for node_id in SAMPLE_NODE_IDS:
    print_leaf_card_preview(node_id, INVENTORY, LEAF_CARD_LOOKUP)


## 2. 一道题进来后，系统怎么推荐知识点

这里用三道题来演示：

- 数列递推转等比
- 向量夹角
- 圆与直线距离

先看系统推荐，再看学生最终确认。

In [ ]:
FIXTURE_DIR = PROJECT_ROOT / 'harness' / 'fixtures' / 'wrong_binder'
CASE_FILES = [
    FIXTURE_DIR / 'binder_case_sequence_shift.json',
    FIXTURE_DIR / 'binder_case_vector_angle.json',
    FIXTURE_DIR / 'binder_case_circle_distance.json',
]
CASE_PAYLOADS = [json.loads(path.read_text(encoding='utf-8')) for path in CASE_FILES]

QUESTION_ID_MAP = {
    'binder_case_sequence_shift': 'wq_user_seq_001',
    'binder_case_vector_angle': 'wq_user_vec_001',
    'binder_case_circle_distance': 'wq_user_circle_001',
}

MANUAL_SELECTIONS = {
    'binder_case_sequence_shift': {
        'primary_node_ids': ['math.数列与不等式.数列.递推数列转化方法.构造等比数列'],
        'secondary_node_ids': ['math.数列与不等式.数列.基础概念.数列递推公式解读'],
        'selection_notes': '学生确认这题主要错在递推转等比。',
    },
    'binder_case_vector_angle': {
        'primary_node_ids': ['math.三角函数_平面向量与解三角形.平面向量.向量夹角公式'],
        'secondary_node_ids': ['math.三角函数_平面向量与解三角形.平面向量.数量积几何意义'],
        'selection_notes': '学生确认核心在夹角公式。',
    },
    'binder_case_circle_distance': {
        'primary_node_ids': ['math.解析几何.直线的方程.点到直线的距离公式'],
        'secondary_node_ids': ['math.解析几何.圆的方程.圆的标准方程'],
        'selection_notes': '学生确认主点在点到直线距离，圆标准方程作辅助。',
    },
}

BINDER = WrongQuestionBinder(enable_embeddings=False)
BINDER_OBJECTS = []
ANNOTATION_RECORDS = []

for index, case in enumerate(CASE_PAYLOADS):
    result = BINDER.bind({'question_payload': case['question_payload']})
    summary = summarize_result(result)
    BINDER_OBJECTS.append({'case': case, 'result': result, 'summary': summary})

    case_id = case['case_id']
    selection = MANUAL_SELECTIONS[case_id]
    selected_node_ids = selection['primary_node_ids'] + selection['secondary_node_ids']

    ANNOTATION_RECORDS.append(
        {
            'annotation_id': f'anno_user_{index + 1:03d}',
            'question_id': QUESTION_ID_MAP[case_id],
            'question_payload': case['question_payload'],
            'recommendation': {'mode': 'heuristic'},
            'student_selection': {
                'selection_status': 'confirmed',
                'selected_node_ids': selected_node_ids,
                'primary_node_ids': selection['primary_node_ids'],
                'secondary_node_ids': selection['secondary_node_ids'],
                'selection_notes': selection['selection_notes'],
                'used_recommendation': True,
            },
            'review_linkage': {
                'review_node_ids': selected_node_ids,
                'review_notes': '确认后的知识点进入复习系统。',
            },
            'created_at': (BASE_TIME + timedelta(minutes=index * 12)).isoformat(),
        }
    )

    print_binder_case(
        case['description'],
        case['question_payload'],
        summary,
        result.binding_record['top_k_candidates'],
        ANNOTATION_RECORDS[-1]['student_selection'],
    )


## 3. 学生开始复习时，第一页会看到什么

这一步把学生最终确认过的知识点和题目，转成复习系统可读的 `review_state`。

然后看两种模式：

- 先知识点后题目
- 先题目后知识点

In [ ]:
REVIEW_STATE = convert_annotations_to_review_state(
    ANNOTATION_RECORDS,
    existing_review_state=None,
    record_id='review_state.user_walkthrough.v1',
    student_id=STUDENT_ID,
    source_batch_id='user_walkthrough_batch_01',
)

EXAMPLE_MAP = load_example_map(EXAMPLES_MD_PATH)

leaf_first = build_review_bundles(
    REVIEW_STATE,
    example_map=EXAMPLE_MAP,
    leaf_card_lookup=LEAF_CARD_LOOKUP,
    now=DEMO_NOW,
    mode='leaf_first',
    bundle_limit=5,
    bundle_question_limit=3,
)
question_first = build_review_bundles(
    REVIEW_STATE,
    example_map=EXAMPLE_MAP,
    leaf_card_lookup=LEAF_CARD_LOOKUP,
    now=DEMO_NOW,
    mode='question_first',
    bundle_limit=5,
    bundle_question_limit=3,
)

print('模式一：先知识点后题目')
for index, bundle in enumerate(leaf_first.review_bundles[:2], start=1):
    print(f'  {index}. {summarize_bundle(bundle)}')

print('\n模式二：先题目后知识点')
for index, bundle in enumerate(question_first.review_bundles[:2], start=1):
    print(f'  {index}. {summarize_bundle(bundle)}')


## 4. 学生点完按钮后，会有什么变化

这里演示两个最典型的反馈：

- 对一个知识点点击“还要练题”
- 对一道题点击“这题没做对”

In [ ]:
SEQ_PRIMARY_NODE_ID = MANUAL_SELECTIONS['binder_case_sequence_shift']['primary_node_ids'][0]
CIRCLE_QUESTION_ID = QUESTION_ID_MAP['binder_case_circle_distance']

base_schedule = build_review_schedule(REVIEW_STATE, now=DEMO_NOW, user_mode='mixed')

node_update = apply_review_action(
    copy.deepcopy(REVIEW_STATE),
    action='node_needs_more_practice',
    target_type='knowledge_point',
    target_id=SEQ_PRIMARY_NODE_ID,
    now=DEMO_NOW,
)
node_schedule = build_review_schedule(node_update.updated_payload, now=DEMO_NOW, user_mode='mixed')

question_update = apply_review_action(
    copy.deepcopy(REVIEW_STATE),
    action='review_result',
    target_type='wrong_question',
    target_id=CIRCLE_QUESTION_ID,
    result='wrong',
    now=DEMO_NOW + timedelta(hours=1),
)
question_schedule = build_review_schedule(question_update.updated_payload, now=DEMO_NOW + timedelta(hours=1), user_mode='mixed')

print('点击“还要练题”之前，知识点前 3：')
for line in summarize_node_ranking(base_schedule, limit=3):
    print('  ' + line)

print('\n点击“还要练题”之后，知识点前 3：')
for line in summarize_node_ranking(node_schedule, limit=3):
    print('  ' + line)

print('\n一道题做错之前，题目前 3：')
for line in summarize_question_ranking(base_schedule, limit=3):
    print('  ' + line)

print('\n一道题做错之后，题目前 3：')
for line in summarize_question_ranking(question_schedule, limit=3):
    print('  ' + line)


## 5. 系统如何形成长期画像

这一步把前面的行为沉淀成事件：

- 绑定了什么知识点
- 学生最后选了什么知识点
- 诊断成什么错因
- 给了什么引导
- 复习时答对还是答错

然后系统会总结出一个轻量长期画像。

In [ ]:
DIAGNOSIS_BY_CASE = {
    'binder_case_sequence_shift': {
        'error_type': 'missing_strategy',
        'confidence': 0.84,
        'reason': '学生知道要从递推式入手，但不会自己提出构造新数列这个中间目标。',
        'evidence': '我知道要从递推式入手，但不知道为什么要构造 a_n+1/2。',
        'coach_mode': 'socratic_light',
        'coach_trap': '学生知道局部知识，但不会先确定中间目标。',
        'coach_prompt': '先让学生明确中间目标，再推进具体变形。',
        'source': 'user_walkthrough_demo',
    },
    'binder_case_vector_angle': {
        'error_type': 'missing_strategy',
        'confidence': 0.79,
        'reason': '学生知道数量积，但不会先做结构转换。',
        'evidence': '我能想到数量积，但不会把向量和的模平方展开到夹角公式。',
        'coach_mode': 'socratic_light',
        'coach_trap': '学生会局部公式，但不会先做结构转换。',
        'coach_prompt': '先点出结构转换，再推进下一步。',
        'source': 'user_walkthrough_demo',
    },
    'binder_case_circle_distance': {
        'error_type': 'concept_gap',
        'confidence': 0.82,
        'reason': '学生不会把张角存在性先转成圆心到直线距离范围。',
        'evidence': '我知道要用点到直线距离，但不知道为什么先转成距离不超过 2。',
        'coach_mode': 'direct_explain',
        'coach_trap': '学生缺少必要概念。',
        'coach_prompt': '先讲清存在性转化，再回到公式。',
        'source': 'user_walkthrough_demo',
    },
}

COACH_BY_CASE = {
    'binder_case_sequence_shift': {
        'content': '我们先别急着算结果，先想这道题最想把递推式改成什么前后项关系。',
        'reply_quality': 'empty',
        'strategy': {'mode': 'socratic_light', 'trap': '学生知道局部知识，但不会先确定中间目标。', 'prompt': '先让学生明确中间目标，再推进具体变形。'},
        'turn_index': 1,
        'done': False,
        'stop_reason': 'continue',
        'reply_analysis': {'quality': 'empty', 'understands': False, 'completed': False, 'reason': '学生还说不出固定比结构。'},
    },
    'binder_case_vector_angle': {
        'content': '这题不是直接套夹角公式，而是先把 a+b=-c 变成长度关系。',
        'reply_quality': 'weak',
        'strategy': {'mode': 'socratic_light', 'trap': '学生会局部公式，但不会先做结构转换。', 'prompt': '先点出结构转换，再只追问一个关键中间量。'},
        'turn_index': 1,
        'done': False,
        'stop_reason': 'continue',
        'reply_analysis': {'quality': 'weak', 'understands': True, 'completed': False, 'reason': '学生已经想到数量积，但不会主动做模平方转换。'},
    },
    'binder_case_circle_distance': {
        'content': '这题关键不是先算 t，而是先把张角存在性变成圆心到直线的距离范围。',
        'reply_quality': 'good',
        'strategy': {'mode': 'direct_explain', 'trap': '学生缺少必要概念。', 'prompt': '先讲清存在性转化，再回到距离公式。'},
        'turn_index': 1,
        'done': False,
        'stop_reason': 'continue',
        'reply_analysis': {'quality': 'good', 'understands': True, 'completed': False, 'reason': '学生接住了关键转化，但还没写完后续不等式。'},
    },
}

REVIEW_RESULT_BY_CASE = {
    'binder_case_sequence_shift': 'wrong',
    'binder_case_vector_angle': 'partial',
    'binder_case_circle_distance': 'wrong',
}

reset_demo_file(MEMORY_EVENTS_JSONL)
memory_review_state = copy.deepcopy(REVIEW_STATE)

for index, record in enumerate(ANNOTATION_RECORDS):
    case_id = CASE_PAYLOADS[index]['case_id']
    question = record['question_payload']
    selection = record['student_selection']
    question_id = record['question_id']
    primary_node_id = selection['primary_node_ids'][0]
    secondary_node_ids = selection['secondary_node_ids']
    selected_node_ids = selection['selected_node_ids']
    occurred_base = BASE_TIME + timedelta(minutes=index * 25)

    append_event(
        binding_result_to_memory_event(
            question_id=question_id,
            question_type=question.get('question_type'),
            primary_node_id=primary_node_id,
            secondary_node_ids=secondary_node_ids,
            candidate_node_ids=selected_node_ids,
            binding_source='student_confirmed',
            occurred_at=occurred_base.isoformat(),
            source_name=question.get('source_name'),
            source_section=question.get('source_section'),
            student_id=STUDENT_ID,
            session_id=SESSION_ID,
            binding_confidence=0.9,
        ),
        path=MEMORY_EVENTS_JSONL,
    )

    append_event(
        student_choice_to_memory_event(
            action_type='select_node',
            target_type='question',
            target_id=question_id,
            occurred_at=(occurred_base + timedelta(minutes=1)).isoformat(),
            question_id=question_id,
            question_type=question.get('question_type'),
            selected_node_ids=selected_node_ids,
            note=selection.get('selection_notes'),
            source_name=question.get('source_name'),
            source_section=question.get('source_section'),
            student_id=STUDENT_ID,
            session_id=SESSION_ID,
        ),
        path=MEMORY_EVENTS_JSONL,
    )

    append_event(
        diagnosis_result_to_memory_event(
            DIAGNOSIS_BY_CASE[case_id],
            question_id=question_id,
            question_type=question.get('question_type'),
            binding={'primary_node_id': primary_node_id, 'secondary_node_ids': secondary_node_ids, 'linked_node_ids': selected_node_ids},
            occurred_at=(occurred_base + timedelta(minutes=3)).isoformat(),
            source_name=question.get('source_name'),
            source_section=question.get('source_section'),
            student_id=STUDENT_ID,
            session_id=SESSION_ID,
        ),
        path=MEMORY_EVENTS_JSONL,
    )

    append_event(
        coach_response_to_memory_event(
            COACH_BY_CASE[case_id],
            question_id=question_id,
            question_type=question.get('question_type'),
            binding={'primary_node_id': primary_node_id, 'secondary_node_ids': secondary_node_ids, 'linked_node_ids': selected_node_ids},
            occurred_at=(occurred_base + timedelta(minutes=6)).isoformat(),
            error_type=DIAGNOSIS_BY_CASE[case_id]['error_type'],
            source_name=question.get('source_name'),
            source_section=question.get('source_section'),
            student_id=STUDENT_ID,
            session_id=SESSION_ID,
        ),
        path=MEMORY_EVENTS_JSONL,
    )

    review_update = apply_review_action(
        memory_review_state,
        action='review_result',
        target_type='wrong_question',
        target_id=question_id,
        result=REVIEW_RESULT_BY_CASE[case_id],
        now=occurred_base + timedelta(days=2),
    )
    memory_review_state = review_update.updated_payload
    append_event(
        review_state_update_to_memory_event(review_update.as_dict(), student_id=STUDENT_ID, session_id=SESSION_ID),
        path=MEMORY_EVENTS_JSONL,
    )

practice_update = apply_review_action(
    memory_review_state,
    action='node_needs_more_practice',
    target_type='knowledge_point',
    target_id=MANUAL_SELECTIONS['binder_case_sequence_shift']['primary_node_ids'][0],
    now=BASE_TIME + timedelta(days=3),
)
memory_review_state = practice_update.updated_payload
append_event(
    review_state_update_to_memory_event(practice_update.as_dict(), student_id=STUDENT_ID, session_id=SESSION_ID),
    path=MEMORY_EVENTS_JSONL,
)

PROFILE = build_profile_from_store(student_id=STUDENT_ID, path=MEMORY_EVENTS_JSONL)
summary = PROFILE['personalization_summary']

print(f'长期主错因：{summary.get("dominant_error_type")}')
print(f'当前画像阶段：{summary.get("memory_stage")}')
print(f'建议讲法：{teaching_mode_label(summary.get("recommended_teaching_mode"))}')
print(f'建议复习模式：{review_mode_label(summary.get("recommended_review_mode"))}')
print('当前最常卡的知识点：')
for item in summary.get('top_recurrent_nodes', [])[:3]:
    print(f'  - {item.get("title")} | 典型问题：{item.get("dominant_error_type")}')
print('画像备注：')
for note in summary.get('notes', [])[:2]:
    print(f'  - {note}')


## 6. 长期画像会怎样影响系统

最后看两个最直观的地方：

- `coach_agent` 的讲解策略
- `review_scheduler` 的排序偏置

In [ ]:
TARGET_RECORD = ANNOTATION_RECORDS[0]
TARGET_QUESTION = TARGET_RECORD['question_payload']
TARGET_REPLY = TARGET_QUESTION['student_answer']

reply_analysis = build_reply_analysis_fallback(
    analyze_student_reply(TARGET_REPLY),
    '本演示只使用本地 fallback，不调用额外模型。',
    completed=infer_completed_from_reply(TARGET_REPLY),
)

base_strategy = get_coach_strategy(
    ErrorType.MISSING_STRATEGY,
    reply_analysis.quality,
    turn_index=0,
    total_turns=4,
    understands=reply_analysis.understands,
    completed=reply_analysis.completed,
)

session_plain = CoachSession(
    problem_text=TARGET_QUESTION['stem'],
    error_type=ErrorType.MISSING_STRATEGY,
    student_profile='学生会从递推式入手，但经常不会自己提出构造辅助数列这种中间目标。',
    initial_student_reply=TARGET_REPLY,
)
session_memory = CoachSession(
    problem_text=TARGET_QUESTION['stem'],
    error_type=ErrorType.MISSING_STRATEGY,
    student_profile='学生会从递推式入手，但经常不会自己提出构造辅助数列这种中间目标。',
    student_memory_profile=PROFILE,
    student_memory_context=build_student_memory_context(PROFILE),
    initial_student_reply=TARGET_REPLY,
)

strategy_plain = apply_student_memory_bias(base_strategy, session_plain)
strategy_memory = apply_student_memory_bias(base_strategy, session_memory)

schedule_plain = build_review_schedule(memory_review_state, now=BASE_TIME + timedelta(days=3, hours=1), user_mode='mixed', student_memory_profile=None)
schedule_memory = build_review_schedule(memory_review_state, now=BASE_TIME + timedelta(days=3, hours=1), user_mode='mixed', student_memory_profile=PROFILE)

print('没有长期画像时：')
print(f'  - 教学模式：{strategy_plain.mode.value}')
print(f'  - 本轮策略：{strategy_plain.prompt}')

print('\n有长期画像时：')
print(f'  - 教学模式：{strategy_memory.mode.value}')
print(f'  - 本轮策略：{strategy_memory.prompt}')
print(f'  - 额外长期提示：{build_student_memory_strategy_note(base_strategy, session_memory)}')

print('\n没有长期画像时，知识点前 3：')
for line in summarize_node_ranking(schedule_plain, limit=3):
    print('  ' + line)

print('\n有长期画像时，知识点前 3：')
for line in summarize_node_ranking(schedule_memory, limit=3):
    print('  ' + line)

print('\n没有长期画像时，题目前 3：')
for line in summarize_question_ranking(schedule_plain, limit=3):
    print('  ' + line)

print('\n有长期画像时，题目前 3：')
for line in summarize_question_ranking(schedule_memory, limit=3):
    print('  ' + line)


## 这份轻展示版适合拿来做什么

- 给别人讲 TeachAgent 现在已经做到哪一步
- 写 app 前先跑一遍，确认主链还通
- 面向用户或家长展示，不会被底层结构淹没

如果要做更技术一点的排查，继续看上一版：

- `notebooks/05_system/teachagent_system_overview.ipynb`